# MERMAID DIAGRAM

```mermaid
flowchart TD
    USER([User Request])

    subgraph SEQ["RootAgent<br/>SequentialAgent"]
        direction TB

        subgraph PAR["ParallelAgent<br/>ParallelWebResearchAgent"]
            direction LR
        
            subgraph BRANCH1[" "]
                R1["SubAgent_1<br/>deeplearning_agent"]
            end
        
            subgraph BRANCH2[" "]
                R2["SubAgent_2<br/>techcrunch_agent"]
            end
        
            subgraph BRANCH3[" "]
                R3["SubAgent_3<br/>arstechnica_agent"]
            end        
        end
        
        STATE[("Session State")]
        
        MERGE["MergerAgent<br/>SynthesisAgent"]

        R1 --> |"deep_learning_result"|STATE
        R2 --> |"techcrunch_result"|STATE
        R3 --> |"ars_technica_result"|STATE
        STATE --> MERGE
    end

    OUTPUT([Structured Answer])

    USER --> SEQ
    MERGE --> OUTPUT

    classDef seq fill:#E8F5E9,stroke:#2E7D32,stroke-width:4px
    classDef parallel fill:#FFF3E0,stroke:#FB8C00,stroke-width:4px
    classDef agent fill:#FFF3E0,stroke:#E65100,stroke-width:4px
    classDef state fill:#FFEBEE,stroke:#C62828,stroke-width:4px
    classDef merger fill:#F3E5F5,stroke:#8E24AA,stroke-width:4px
    classDef io fill:#E3F2FD,stroke:#1565C0,stroke-width:4px


    class SEQ seq
    class PAR parallel
    class R1,R2,R3 agent
    class STATE state
    class MERGE merger
    class USER,OUTPUT io
```

# IMPORTS

In [1]:
from google.adk.agents import Agent
from google.adk.agents.parallel_agent import ParallelAgent
from google.adk.agents.sequential_agent import SequentialAgent
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools.tool_context import ToolContext
from google.adk.tools.base_tool import BaseTool
from google.genai import types
from typing import Dict, Any
from dotenv import load_dotenv
from google.adk.models.lite_llm import LiteLlm
import uuid

load_dotenv()

# MODEL = "gemini-3.5-flash-lite"
MODEL = LiteLlm(model="ollama_chat/gemma4:cloud")
MODEL_LCW = LiteLlm(model="ollama_chat/nemotron-3-nano:30b-cloud")

# TOOL DEFINITION

In [2]:
import concurrent.futures
from typing import List, Dict
from datetime import datetime, timezone
from bs4 import BeautifulSoup
from curl_cffi import requests
from ddgs import DDGS

def _extract_pub_date(soup: BeautifulSoup) -> str:
    """
    Attempts to extract the publication date from common HTML5 and meta tags.
    Returns the raw string date if found, otherwise 'Unknown'.
    """
    # Check standard OpenGraph and Schema.org meta tags commonly used by news sites
    meta_tags = [
        ('property', 'article:published_time'),
        ('property', 'og:article:published_time'),
        ('name', 'pubdate'),
        ('name', 'date'),
        ('name', 'dc.date'),
        ('name', 'creation_date')
    ]
    
    for attr, val in meta_tags:
        meta_date = soup.find('meta', attrs={attr: val})
        if meta_date and meta_date.get('content'):
            return meta_date.get('content').strip()
            
    # Check for HTML5 <time> tag as a fallback
    time_tag = soup.find('time')
    if time_tag and time_tag.get('datetime'):
        return time_tag.get('datetime').strip()
        
    return "Unknown"

def _parse_date_for_sorting(date_str: str) -> datetime:
    """
    Converts a date string into an offset-aware datetime object (UTC) for reliable sorting.
    If the date is missing or malformed, it returns a minimum datetime in UTC, 
    so those articles naturally fall to the bottom of the list.
    """
    # Create a base minimum date explicitly bound to the UTC timezone
    min_date_utc = datetime.min.replace(tzinfo=timezone.utc)
    
    if date_str == "Unknown" or not date_str:
        return min_date_utc
        
    # Standardize string for Python's fromisoformat
    clean_date = date_str.replace("Z", "+00:00")
    
    try:
        dt = datetime.fromisoformat(clean_date)
        
        # If the parsed date is naive (no timezone attached by the website),
        # we must attach a UTC timezone to it to prevent TypeError during sorting.
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
            
        return dt
    except ValueError:
        # If the date format is completely unparseable, return the minimum date
        return min_date_utc

def _fetch_and_parse_article(session: requests.Session, url: str, fallback_snippet: str, timeout: int = 20) -> Dict[str, str]:
    """
    Worker function to fetch and parse a single article.
    Designed to be run safely in parallel threads.
    """
    try:
        response = session.get(url, timeout=timeout)
        
        if response.status_code != 200:
            return {
                "content": f"[HTTP {response.status_code}] Fallback Snippet: {fallback_snippet}",
                "pub_date": "Unknown"
            }
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # 1. Extract the publication date before modifying the DOM
        pub_date = _extract_pub_date(soup)
        
        # 2. Safely clean the HTML tree by removing noisy elements
        for element in soup(["script", "style", "nav", "header", "footer"]):
            element.decompose()
            
        # 3. Extract text from ALL paragraph tags directly
        paragraphs = soup.find_all('p')
        
        # Join paragraphs, filtering out empty strings or tiny UI artifacts
        extracted_text = "\n".join(
            p.get_text(separator=' ', strip=True) 
            for p in paragraphs 
            if len(p.get_text(strip=True)) > 15
        )
        
        if not extracted_text:
            return {
                "content": f"[Empty Content] Fallback Snippet: {fallback_snippet}",
                "pub_date": pub_date
            }
            
        return {
            "content": extracted_text,
            "pub_date": pub_date
        }

    except Exception as e:
        print(f"Failed to fetch content for {url}: {e}")
        return {
            "content": f"[Connection Error] Fallback Snippet: {fallback_snippet}",
            "pub_date": "Unknown"
        }

def duckduckgo_search(query: str, max_results: int = 5, domain: str = "agi.it") -> List[Dict[str, str]]:
    """
    Searches for news articles on a specific domain using DuckDuckGo.
    Extracts the full content and publication date concurrently using multithreading.
    Returns the results sorted in reverse chronological order (newest first).
    
    Args:
        query (str): The search keyword(s).
        max_results (int): The maximum number of articles to extract.
        domain (str): The domain to restrict the search to.

    Returns:
        List[Dict[str, str]]: A list of dictionaries containing title, link, pub_date, and content.
    """
    search_query = f"{query} site:{domain}"
    results = []
    
    try:
        ddg_results = list(DDGS().text(search_query, max_results=max_results))
    except Exception as e:
        print(f"Error fetching DuckDuckGo results: {e}")
        return []

    # Reusing the TCP connection speeds up the scraping significantly
    with requests.Session(impersonate="chrome120") as session:
        session.headers.update({
            "Referer": "https://duckduckgo.com/",
            "Accept-Language": "en-US,en;q=0.9",
        })
        
        # Fetch articles concurrently
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_results) as executor:
            future_to_item = {}
            
            for item in ddg_results:
                real_url = item.get("href", "")
                if not real_url:
                    continue
                    
                fallback_snippet = item.get("body", "No snippet available")
                future = executor.submit(_fetch_and_parse_article, session, real_url, fallback_snippet)
                future_to_item[future] = item
                
            for future in concurrent.futures.as_completed(future_to_item):
                original_item = future_to_item[future]
                title = original_item.get("title", "No title")
                link = original_item.get("href", "")
                
                try:
                    data = future.result()
                    article_content = data["content"]
                    pub_date = data["pub_date"]
                except Exception as exc:
                    print(f"Thread generated an exception for {link}: {exc}")
                    article_content = f"[Thread Error] Fallback Snippet: {original_item.get('body', '')}"
                    pub_date = "Unknown"
                    
                results.append({
                    "title": title,
                    "link": link,
                    "pub_date": pub_date,
                    "content": article_content
                })
                
    # Sort the results: Reverse chronological order (Newest articles first)
    # The _parse_date_for_sorting function now correctly aligns all timezones
    results.sort(key=lambda x: _parse_date_for_sorting(x["pub_date"]), reverse=True)
    
    return results

# CALLBACK FUNCTIONS DEFINITION

In [3]:
# def get_before_tool_callback(agent_name: str):
#     def before_tool(tool, args, ctx):
#         print(f"[{agent_name}] 🛠️ Tool Call: {getattr(tool, 'name', 'tool')} | Args: {args}", flush=True)
#     return before_tool

# def get_after_tool_callback(agent_name: str):
#     def after_tool(tool, args, ctx, result):
#         print(f"[{agent_name}] 📋 Tool Execution Completed", flush=True)
#     return after_tool




def before_tool(tool: BaseTool, args: Dict[str, Any], tool_context: ToolContext):
    # tool.name
    print(f"[{tool_context.agent_name}] 🛠️ Callback Before Tool Call: {getattr(tool, 'name', 'tool')} | Args: {args}", flush=True)
    #print(f"[{tool_context.node.name}] 🛠️ Callback Before Tool Call: {getattr(tool, 'name', 'tool')} | Args: {args}", flush=True)
    return None

def after_tool(tool: BaseTool, args: Dict[str, Any], tool_context: ToolContext, tool_response: Any):
    # tool.name
    print(f"[{tool_context.agent_name}] 🛠️ Callback After Tool Call: {getattr(tool, 'name', 'tool')}", flush=True)
    # print(f"[{tool_context.node.name}] 🛠️ Callback After Tool Call: {getattr(tool, 'name', 'tool')}", flush=True)
    return None

# DEEP LEARNING AI SUBAGENT

In [4]:
subagent_1 = Agent(
    name="deeplearning_agent",
    model=MODEL,
    description="Professional search assistant with DuckDuckGo Search capabilities.",
    instruction="Answer questions using the DuckDuckGo Search Tool *only with* the 'deeplearning.ai' domain. Do not answer with your own knowledge. Do not summarize results. Report the entire object returned by the tool.",
    tools=[duckduckgo_search],
    before_tool_callback=before_tool,
    after_tool_callback=after_tool,
    # Store result in state for the merger agent
    output_key="deep_learning_result",
)

# TECHCRUNCH SUBAGENT

In [5]:
subagent_2 = Agent(
    name="techcrunch_agent",
    model=MODEL,
    description="Professional search assistant with DuckDuckGo Search capabilities.",
    instruction="Answer questions using the DuckDuckGo Search Tool *only with* the 'https://techcrunch.com/category/artificial-intelligence/' domain. Do not answer with your own knowledge. Do not summarize results. Report the entire object returned by the tool.",
    tools=[duckduckgo_search],
    before_tool_callback=before_tool,
    after_tool_callback=after_tool,
    # Store result in state for the merger agent
    output_key="techcrunch_result",
)

# ARS TECHNICA AI SUBAGENT

In [6]:
subagent_3 = Agent(
    name="arstechnica_agent",
    model=MODEL,
    description="Professional search assistant with DuckDuckGo Search capabilities.",
    instruction="Answer questions using the DuckDuckGo Search Tool *only with* the 'arstechnica.com/ai' domain. Do not answer with your own knowledge. Do not summarize results. Report the entire object returned by the tool.",
    tools=[duckduckgo_search],
    before_tool_callback=before_tool,
    after_tool_callback=after_tool,
    # Store result in state for the merger agent
    output_key="ars_technica_result",
)

# PARALLEL AGENT

In [7]:
parallel_research_agent = ParallelAgent(     
    name="ParallelWebResearchAgent",     
    sub_agents=[subagent_1, subagent_2, subagent_3],     
    description="Runs multiple research agents in parallel to gather information." 
)

/tmp/ipykernel_1677209/2613729482.py:1: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  parallel_research_agent = ParallelAgent(


# MERGER AGENT

In [8]:
merger_agent = Agent(     
    name="SynthesisAgent",     
    model=MODEL,
    instruction="""
         You are an AI Assistant responsible for combining research findings into a structured report.    
         Your primary task is to synthesize the following research summaries, clearly attributing findings to their source. Structure your response using headings for each search.    
         **Crucially: Your entire response MUST be grounded *exclusively* on the information provided in the 'Input Results' below. Do NOT add any external knowledge, facts, or details not present in these specific summaries.**
    
         **Input Results:**
    
         *   **DEEP LEARNING:**
             {deep_learning_result}
    
         *   **TECHCRUNCH:**
             {techcrunch_result}

         *   **ARS TECHNICA:**
             {ars_technica_result}
    
         ## Output Format

        The response must strictly follow a structure similar to that used in scientific publications.
        
        ### 1. Main Body
        
        - The main body must be from 5 up to 15 lines.
        - The text must be concise, informative, and based exclusively on the sources actually consulted.
        - Every factual statement derived from a source must be accompanied, at the exact point where it is used, by the corresponding citation number.
        - Citations must be indicated exclusively using the format `[N]`, where N is the sequential number assigned to the source based on its first appearance in the text.
        - If the same source is used multiple times, it must always retain the same citation number.
        - Do not create citations that do not correspond to a source actually consulted.
        - Do not include links directly in the main body.
        - Do not use citation formats other than `[N]`.
        
        Example:
        
        According to the latest data, the market grew by 12% during the first half of the year [1]. This growth was primarily driven by the technology sector [2][3].
        
        ### 2. Bibliography / Sources
        
        After the main body, include a section containing all sources used.
        Sources must be ordered exclusively according to their first appearance in the text, NOT alphabetically, chronologically, or by author.
        For each source, use exactly the following structure:
        
        [1] YYYY Month DD - Title
               <link>
        
        [2] YYYY Month DD - Title
               <link>
        
        ### 3. Bibliographic Information Rules
        
        For each citation, provide the following information, in this exact order:
        1. `pub_date`
        2. `title`
        3. `link`
        
        The date must use the following format:
        
        `YYYY Month DD`
        
        Example:
        
        `2026 August 14`
        
        The title must be reported in the original form used by the source, without modifying or translating it.
        The link must point to the actual page used as the source.
        
        ### 4. Numbering Rules
        
        - The first source used in the text is `[1]`.
        - The second new source used is `[2]`.
        - And so on.
        - The number is assigned when the source first appears in the text.
        - If a previously cited source is used again, do not assign it a new number.
        - The final bibliography must follow exactly the same numbering.
        
        ### 5. Completeness and Accuracy
        
        - Every `[N]` appearing in the text must have a corresponding entry in the bibliography.
        - Every entry in the bibliography must have been cited at least once in the text.
        - Do not include sources that were consulted but not used in the text.
        - Do not invent `pub_date`, `title`, or `link`.
        - If the publication date is unavailable, use `N/A` instead of inventing a date.
        - If the title is unavailable, use `N/A`.
        - Use the most direct link possible to the original source.
        
        ### 6. Final Validation
        
        Before returning the response, internally verify that:
        
        - the main body lenght is comprised between 5 and 15 lines;
        - all source-based factual statements have a `[N]` citation;
        - the numbering is sequential and contains no gaps;
        - each `[N]` in the text corresponds to exactly one source;
        - the bibliography is ordered according to the sources' first appearance in the text;
        - every bibliography entry contains exactly `pub_date`, `title`, and `link`, in that order;
        - no uncited sources are included in the bibliography.

    
         Output *only* the structured report following this format. Do not include introductory or concluding phrases outside this structure, and strictly adhere to using only the provided input summary content.
         """,     
    description="Combines research findings from parallel agents into a structured, cited report, strictly grounded on provided inputs.",
)

# ORCHESTRATOR ROOT AGENT

In [9]:
# This is the main agent that will be run. It first executes the ParallelAgent 
# to populate the state, and then executes the MergerAgent to produce the final output. 
root_agent = SequentialAgent(     
    name="root_agent",   
    sub_agents=[parallel_research_agent, merger_agent],
    description="Coordinates parallel research and synthesizes the results."
)

/tmp/ipykernel_1677209/3804017888.py:3: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  root_agent = SequentialAgent(


# SETUP RUNNER AND SESSION SERVICE

In [10]:
session_service = InMemorySessionService()

# Define constants for identifying the interaction context
APP_NAME = "web_searcher_app"
USER_ID = "user_1"
#SESSION_ID = "session_001"
SESSION_ID = f"session_{uuid.uuid4().hex[:8]}"

# Create the specific session where the conversation will happen
session = await session_service.create_session(    
    app_name = APP_NAME,    
    user_id = USER_ID,    
    session_id = SESSION_ID,
)
print(f"Session created: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

# --- OR if running as a standard Python script ---
# from google.adk.sessions import Session
#
# async def init_session(app_name:str,user_id:str,session_id:str) -> Session:
#     session = await session_service.create_session(
#         app_name=app_name,
#         user_id=user_id,
#         session_id=session_id
#     )
#     print(f"Session created: App='{app_name}', User='{user_id}', Session='{session_id}'")
#     return session
#
# session = asyncio.run(init_session(APP_NAME,USER_ID,SESSION_ID))

# --- Runner ---
# Key Concept: Runner orchestrates the agent execution loop.
runner = Runner(    
    agent = root_agent,
    app_name = APP_NAME,
    session_service = session_service,
)
print(f"Runner created for agent '{runner.agent.name}'.")

Session created: App='web_searcher_app', User='user_1', Session='session_3b378ee7'
Runner created for agent 'root_agent'.


# FUNCTION FOR AGENT INTERACTION

In [11]:
# async def call_agent_async(query: str, runner, user_id, session_id):
#   """Sends a query to the agent and prints the final response."""
#   print(f"\n>>> User Query: {query}")

#   # Prepare the user's message in ADK format
#   content = types.Content(role='user', parts=[types.Part(text=query)])

#   final_response_text = "Agent did not produce a final response." # Default

#   # Key Concept: run_async executes the agent logic and yields Events.
#   # We iterate through events to find the final answer.
#   async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
#       # You can uncomment the line below to see *all* events during execution
#       # print(f"  [Event] Author: {event.author}, Type: {type(event).__name__}, Final: {event.is_final_response()}, Content: {event.content}")

#       # Key Concept: is_final_response() marks the concluding message for the turn.
#       if event.is_final_response():
#           if event.content and event.content.parts:
#              # Assuming text response in the first part
#              final_response_text = event.content.parts[0].text
#           elif event.actions and event.actions.escalate: # Handle potential errors/escalations
#              final_response_text = f"Agent escalated: {event.error_message or 'No specific message.'}"
#           # Add more checks here if needed (e.g., specific error codes)
#           break # Stop processing events once the final response is found

#   print(f"<<< Agent Response: {final_response_text}")



# async def call_agent_async(query: str, runner, user_id, session_id):
#     """Sends a query to the agent and prints intermediate subagent steps and the final response."""
#     print(f"\n>>> User Query: {query}\n")

#     # Prepare the user's message in ADK format
#     content = types.Content(role='user', parts=[types.Part(text=query)])
#     final_response_text = "Agent did not produce a final response."

#     # Iterate through events to find subagent actions, tool calls, and the final answer
#     async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
        
#         # Get the name of the subagent (eg. google_search_agent)
#         author = getattr(event, 'author', 'System')
        
#         if event.content and event.content.parts:
#             # 1. The agent calls the tool
#             for part in event.content.parts:
#                 if hasattr(part, 'function_call') and part.function_call:
#                     print(f"[{author}] 🛠️ Tool Call with parameters:")
#                     print(f"    Tool: {part.function_call.name}")
#                     print(f"    Args: {part.function_call.args}")
                    
#                 # 2. The tool returns an answer
#                 elif hasattr(part, 'function_response') and part.function_response:
#                     print(f"[{author}] 📋 Tool Execution Completed ({part.function_response.name})")
#                     print(f"    Output: {part.function_response.response}")
                    
#                 # 3. L'agente genera del testo (ragionamento intermedio o risposta finale del subagente)
#                 elif part.text:
#                     # Stampiamo l'output parziale solo se l'evento NON è quello finale (per evitare doppioni)
#                     if not event.is_final_response():
#                         print(f"\n[{author}] 💬 Output Agent Execution:")
#                         print(f"    {part.text}\n")
#                         print("-" * 60)
                        
#         # # 4. Verifica se abbiamo raggiunto la risposta finale dell'Orchestrator
#         # if event.is_final_response():
#         #     if event.content and event.content.parts:
#         #         final_response_text = event.content.parts[0].text
#         #     elif hasattr(event, 'actions') and getattr(event.actions, 'escalate', False):
#         #         final_response_text = f"Agent escalated: {getattr(event, 'error_message', 'No specific message.')}"
#         #     break

#         # 4. Verifica se abbiamo raggiunto la risposta finale dell'Orchestrator
#         if getattr(event, 'is_final_response', lambda: False)():
#             if event.content and event.content.parts:
#                 final_response_text = event.content.parts[0].text
#             elif hasattr(event, 'actions') and getattr(event.actions, 'escalate', False):
#                 final_response_text = f"Agent escalated: {getattr(event, 'error_message', 'No specific message.')}"


#     print("\n" + "="*80)
#     print("\n" + "="*80)
#     print(f"<<< FINAL ORCHESTRATOR RESPONSE:\n{final_response_text}")
#     print("="*80 + "\n")



# async def call_agent_async(query: str, runner, user_id, session_id):
#     print(f"\n>>> User Query: {query}\n")
#     content = types.Content(role='user', parts=[types.Part(text=query)])
    
#     final_response_text = "Il SynthesisAgent non ha prodotto una risposta (possibile errore interno)."

#     try:
#         async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
#             author = getattr(event, 'author', 'System')
            
#             # Stampa le chiamate ai tool dei subagenti in tempo reale
#             if event.content and event.content.parts:
#                 for part in event.content.parts:
#                     if hasattr(part, 'function_call') and part.function_call:
#                         print(f"[{author}] 🛠️ Tool Call: {part.function_call.name} | Args: {part.function_call.args}")
#                     elif hasattr(part, 'function_response') and part.function_response:
#                         print(f"[{author}] 📋 Tool Execution Completed")
            
#             # CATTURIAMO ESCLUSIVAMENTE LA RISPOSTA DEL SYNTHESIS AGENT
#             if getattr(event, 'is_final_response', lambda: False)():
#                 if author in ["SynthesisAgent", "root_agent"]:
#                     if event.content and event.content.parts:
#                         final_response_text = event.content.parts[0].text
#                     elif hasattr(event, 'actions') and getattr(event.actions, 'escalate', False):
#                         final_response_text = f"Errore nel SynthesisAgent: {getattr(event, 'error_message', 'No message')}"
                
#     except Exception as e:
#         print(f"\n[ERRORE DI ESECUZIONE DEL GRAFO]: {e}")

#     print("\n" + "="*80)
#     print(f"<<< FINAL SYNTHESIS RESPONSE:\n{final_response_text}")
#     print("="*80 + "\n")


# async def call_agent_async(query: str, runner, user_id, session_id):
#     """
#     Sends a query to the agent and prints intermediate subagent steps,
#     debug outputs, and the final synthesized response.
#     """
#     print(f"\n>>> User Query: {query}\n")
    
#     # Prepare the user's message in ADK format
#     content = types.Content(role='user', parts=[types.Part(text=query)])
#     final_response_text = "The SynthesisAgent did not produce a response (possible internal error or loop hanging)."

#     try:
#         # Iterate through events to find subagent actions, tool calls, and the final answer
#         async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
            
#             # Get the name of the agent generating the event
#             author = getattr(event, 'author', 'System')
            
#             # 1. Print real-time tool calls and responses
#             if event.content and event.content.parts:
#                 for part in event.content.parts:
#                     if hasattr(part, 'function_call') and part.function_call:
#                         print(f"[{author}] 🛠️ Tool Call: {part.function_call.name} | Args: {part.function_call.args}")
#                     elif hasattr(part, 'function_response') and part.function_response:
#                         print(f"[{author}] 📋 Tool Execution Completed")
            
#             # 2. Check if this is a final response of ANY agent in the workflow
#             if getattr(event, 'is_final_response', lambda: False)():
                
#                 # If the author is the final merger agent, save its response to print at the very end
#                 if author in ["SynthesisAgent", "root_agent"]:
#                     if event.content and event.content.parts:
#                         final_response_text = event.content.parts[0].text
#                     elif hasattr(event, 'actions') and getattr(event.actions, 'escalate', False):
#                         final_response_text = f"Error in SynthesisAgent: {getattr(event, 'error_message', 'No message')}"
                        
#                 # Otherwise, it's a subagent finishing its parallel task. Print its output for debug!
#                 else:
#                     print(f"\n{'#'*20} ONLY FOR DEBUG PURPOSES {'#'*20}")
#                     print(f"[{author}] finished its execution and produced the following result:")
#                     if event.content and event.content.parts:
#                         print(event.content.parts[0].text)
#                     print(f"{'#'*65}\n")
                    
#     except Exception as e:
#         print(f"\n[GRAPH EXECUTION ERROR]: {e}")

#     # 3. Print the final synthesized response
#     print("\n" + "="*120)
#     print(f"<<< FINAL SYNTHESIS RESPONSE:\n{final_response_text}")
#     print("="*120 + "\n")



async def call_agent_async(query: str, runner, user_id, session_id):
    content = types.Content(role='user', parts=[types.Part(text=query)])
    
    final_response = "The SynthesisAgent did not produce a response (possible internal error or hanging)."
    agent_results = []

    try:
        async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
            author = getattr(event, 'author', 'System')
            
            # Controlliamo solo se è una risposta finale (di qualsiasi agente)
            if getattr(event, 'is_final_response', lambda: False)():
                
                if author in ["SynthesisAgent", "root_agent"]:
                    # Risposta finale dell'Orchestrator
                    if event.content and event.content.parts:
                        final_response = event.content.parts[0].text
                    elif hasattr(event, 'actions') and getattr(event.actions, 'escalate', False):
                        final_response = f"Error in SynthesisAgent: {getattr(event, 'error_message', 'No message')}"
                else:
                    # Salviamo i risultati intermedi dei subagenti nella lista di dizionari
                    if event.content and event.content.parts:
                        agent_results.append({
                            "agent": author,
                            "result": event.content.parts[0].text
                        })
                        
    except Exception as e:
        final_response = f"[GRAPH EXECUTION ERROR]: {e}"

    return final_response, agent_results

# AGENT INTERACTION

In [12]:
# async def run_conversation():    
#     await call_agent_async("What are the latest news about Nvidia?",                                       
#     runner=runner,                                       
#     user_id=USER_ID,                                       
#     session_id=SESSION_ID
# )

query = "What are the latest news about Nvidia?"
print(f"\n>>> User Query: {query}\n", flush=True)

final_response, agent_results = await call_agent_async(
    query=query,                                       
    runner=runner,                                       
    user_id=USER_ID,                                       
    session_id=SESSION_ID
)


>>> User Query: What are the latest news about Nvidia?



/home/eljamba/miniconda3/lib/python3.14/site-packages/google/adk/tools/function_tool.py:96: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


[deeplearning_agent] 🛠️ Callback Before Tool Call: duckduckgo_search | Args: {'domain': 'deeplearning.ai', 'query': 'Nvidia'}


[deeplearning_agent] 🛠️ Callback After Tool Call: duckduckgo_search


[techcrunch_agent] 🛠️ Callback Before Tool Call: duckduckgo_search | Args: {'domain': 'https://techcrunch.com/category/artificial-intelligence/', 'query': 'Nvidia'}


/tmp/ipykernel_1677209/194659469.py:77: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(response.content, 'html.parser')


[techcrunch_agent] 🛠️ Callback After Tool Call: duckduckgo_search


[arstechnica_agent] 🛠️ Callback Before Tool Call: duckduckgo_search | Args: {'domain': 'arstechnica.com/ai', 'query': 'Nvidia'}


[arstechnica_agent] 🛠️ Callback After Tool Call: duckduckgo_search


In [13]:
print("\n" + "="*80)
print("<<< FINAL SYNTHESIS RESPONSE:")
print(final_response)
print("="*80 + "\n")


<<< FINAL SYNTHESIS RESPONSE:
Nvidia is reportedly pursuing the acquisition of AI model repository Hugging Face for $12.9 billion to fortify its industry integration [1]. The company is also preparing "NemoClaw," an open-source AI agent platform designed to compete with OpenClaw [2]. In hardware, Nvidia launched the DGX Spark (originally previewed as Project DIGITS), a desktop AI computer starting at $3,999 with 128GB of unified memory capable of running models up to 200 billion parameters [3][4][5]. Additionally, the more powerful DGX Station utilizes the GB300 Grace Blackwell Ultra Desktop Superchip [4]. Regarding software, Nvidia released the Nemotron 3 Super 120B-A12B open-weights model and plans to invest $26 billion over five years into open-weights model development [6]. Furthermore, Nvidia is applying AI to five stages of chip design, including verifying designs and laying out components, with reinforcement learning agents producing measurably superior circuits [7].

**Bibliog

In [18]:
print("\n" + "#"*20 + " SINGLE AGENT RESULTS " + "#"*20)
for res in agent_results:
    print(f"\n[{res['agent']}] returned data:")
    print(res['result']) 
    print("-" * 60)
    print("\n\n")


#################### SINGLE AGENT RESULTS ####################

[arstechnica_agent] returned data:
```json
[
  {
    "title": "Report: Nvidia to acquire AI model repository Hugging Face for $13 billion - Ars Technica",
    "link": "https://arstechnica.com/ai/2026/08/report-nvidia-to-acquire-ai-model-repository-hugging-face-for-13-billion/",
    "pub_date": "2026-08-27T19:55:22+00:00",
    "content": "Nvidia is reportedly moving forward to acquire Hugging Face for $12.9 billion. The acquisition could help the hardware giant expand and fortify its deep integration with the wider AI industry.\nAmong other things, Hugging Face is a cloud repository for AI models, similar in some respects to what GitHub is for conventional computer software. Developers and researchers search for models that meet certain criteria, download and run them, and fine-tune them into different variants that then get uploaded back up to Hugging Face.\nThe deal is not finalized, so it could still run into trouble ah